# Local Eval Harness — Held-out Validation Slice

**Purpose:** score adapter quality on a held-out slice of `train.csv` *without* burning a daily submission slot. Lets us iterate on training/prompting between leaderboard submissions.

**Project:** this repo — see `PLAN.md` and `EXPERIMENTS.md`.

**Discipline:**
- Fixed split: held-out = last 100 rows of `train.csv` sorted by `id` (deterministic, reproducible)
- Same held-out slice across every eval run, so deltas are comparable
- v1 metric: **exact-match accuracy** on extracted final answer (we'll refine extraction/scoring as we learn the comp's actual grading rules)
- Seed = 42
- Default: load **untrained** LoRA adapter (= base model). Future: load a trained adapter from a Kaggle Dataset before this point.

**Required Kaggle inputs (declared in kernel-metadata.json):**
- Competition data + Model + Utility scripts (same as baseline)

**This kernel does NOT submit to leaderboard.** It produces `eval_results.csv` in `/kaggle/working/` with per-row predictions + scores.

In [ ]:
# Cell 1: Config — change these to vary the eval
import os
import random

import numpy as np
import torch

SEED = 42
HOLDOUT_N = 100              # how many problems to eval on (fixed split: LAST n by id sort)
MAX_NEW_TOKENS = 1024        # generation budget per problem
TEMPERATURE = 0.0            # 0 = greedy decoding (deterministic)
PROMPT_TEMPLATE = "{prompt}"  # v1: pass the prompt as-is. Try variations later (CoT prefix, etc.)

# --- seeds ---
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Config: HOLDOUT_N={HOLDOUT_N}, SEED={SEED}, TEMP={TEMPERATURE}, MAX_NEW_TOKENS={MAX_NEW_TOKENS}")

In [ ]:
# Cell 2: Load base model + (untrained) LoRA adapter
import site

# Auto-discover python_packages from attached utility scripts (see baseline notebook)
ROOTS_TO_SCAN = [
    "/kaggle/input/notebooks",
    "/kaggle/input/utility-scripts",
    "/kaggle/usr/lib/notebooks",
    "/kaggle/lib",
]
found = []
for utility_root in ROOTS_TO_SCAN:
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _files in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)
            found.append(root)
            print(f"sitedir+: {root}")

import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
LORA_RANK = 32

print(f"Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.eval()

# Untrained LoRA = base model behavior at inference (B-matrix is zero-init)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
print("Model + untrained adapter loaded.")

In [ ]:
# Cell 2.5: Blackwell ptxas workaround
# The ryanholbrook/nvidia-utility-script ships ptxas-blackwell (PTX assembler for
# Blackwell GPUs) but Kaggle mounts it read-only without exec permission, so
# Triton's runtime kernel compile fails when the model tries to use Mamba's
# CUDA fast path.
#
# Two-pronged fix:
#   (a) Copy ptxas-blackwell to /tmp/ where we can chmod +x it; point Triton at it.
#   (b) If that still fails at compile time, disable Mamba fast path entirely —
#       falls back to slower pure-PyTorch path. Slower but always works.
import os
import shutil
import stat
import sys

PTXAS_SRC_CANDIDATES = [
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
]
PTXAS_DST = "/tmp/ptxas-blackwell"

ptxas_fixed = False
for src in PTXAS_SRC_CANDIDATES:
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(
                PTXAS_DST,
                stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH,
            )
        # Triton reads several env-var names depending on version; set them all.
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        print(f"ptxas workaround: copied {src} -> {PTXAS_DST} (+x)")
        ptxas_fixed = True
        break

if not ptxas_fixed:
    print("ptxas-blackwell not found in expected locations; relying on fast-path-disable fallback")

# Belt-and-suspenders: also disable Mamba fast path. If ptxas copy worked, this is
# unnecessary; if it didn't, this prevents the Triton compile from being attempted.
# Slow path is pure PyTorch — produces identical numerical output, just slower.
patched = 0
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        mod.is_fast_path_available = False  # type: ignore[attr-defined]
        patched += 1
        print(f"Disabled Mamba fast path in {name}")
if patched == 0:
    print("WARNING: modeling_nemotron_h module not in sys.modules yet — "
          "fast path may still attempt Triton compile. If the next cell errors, "
          "rerun this cell after the model has been loaded.")

In [ ]:
# Cell 3: Load held-out validation slice from train.csv
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV)
print(f"Train: {df.shape}")

# Fixed split: sort by id, take last HOLDOUT_N rows. Deterministic across runs.
df_sorted = df.sort_values("id").reset_index(drop=True)
holdout = df_sorted.tail(HOLDOUT_N).reset_index(drop=True)
print(f"Held-out: {holdout.shape} (last {HOLDOUT_N} by id sort)")
print(f"First held-out id: {holdout.iloc[0]['id']}, last: {holdout.iloc[-1]['id']}")

In [ ]:
# Cell 4: Run inference on each held-out problem
import time

results = []
t_start = time.time()

for i, row in holdout.iterrows():
    pid = row["id"]
    prompt = PROMPT_TEMPLATE.format(prompt=row["prompt"])
    gold = str(row["answer"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(TEMPERATURE > 0),
            temperature=TEMPERATURE if TEMPERATURE > 0 else 1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Strip the prompt prefix from the generation
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = full[len(prompt):] if full.startswith(prompt) else full

    results.append({
        "id": pid,
        "prompt": prompt,
        "gold": gold,
        "completion": completion,
    })

    if (i + 1) % 10 == 0 or i == len(holdout) - 1:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        remaining = (len(holdout) - i - 1) / rate if rate > 0 else 0
        print(f"  {i+1}/{len(holdout)} done ({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)")

print(f"Inference complete in {time.time() - t_start:.0f}s")

In [ ]:
# Cell 5: Score predictions
# v1 scoring: two metrics
#   - contains_match: gold answer appears anywhere in completion (permissive)
#   - exact_match: completion stripped of whitespace == gold stripped of whitespace (strict)
# We'll likely refine to match the comp's actual extractor once we know it.

import re

def contains_match(completion: str, gold: str) -> bool:
    return gold.strip() in completion

def exact_match(completion: str, gold: str) -> bool:
    return completion.strip() == gold.strip()

def last_line_match(completion: str, gold: str) -> bool:
    lines = [ln.strip() for ln in completion.strip().split("\n") if ln.strip()]
    if not lines:
        return False
    return lines[-1] == gold.strip()

for r in results:
    r["contains_match"] = contains_match(r["completion"], r["gold"])
    r["exact_match"] = exact_match(r["completion"], r["gold"])
    r["last_line_match"] = last_line_match(r["completion"], r["gold"])

n = len(results)
contains_acc = sum(r["contains_match"] for r in results) / n
exact_acc = sum(r["exact_match"] for r in results) / n
last_line_acc = sum(r["last_line_match"] for r in results) / n

print(f"\n=== Held-out eval (n={n}) ===")
print(f"  contains_match : {contains_acc:.3f}  (gold appears anywhere in completion)")
print(f"  exact_match    : {exact_acc:.3f}  (whole completion == gold)")
print(f"  last_line_match: {last_line_acc:.3f}  (last non-empty line == gold)")

In [ ]:
# Cell 6: Save eval results to /kaggle/working for download/inspection
import json

out_df = pd.DataFrame(results)
out_path = "/kaggle/working/eval_results.csv"
out_df.to_csv(out_path, index=False)
print(f"Wrote {out_path} ({len(out_df)} rows, {out_df.shape[1]} cols)")

summary = {
    "holdout_n": n,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "prompt_template": PROMPT_TEMPLATE,
    "adapter": "untrained (= base model)",
    "contains_match": contains_acc,
    "exact_match": exact_acc,
    "last_line_match": last_line_acc,
}
with open("/kaggle/working/eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# Cell 7: Print 5 failure examples for failure-mode analysis (per our discipline)
failures = [r for r in results if not r["contains_match"]][:5]
print(f"\n=== 5 failure examples (out of {sum(1 for r in results if not r['contains_match'])} total failures) ===\n")
for i, r in enumerate(failures, 1):
    print(f"--- Failure {i} (id={r['id']}) ---")
    print(f"PROMPT (last 200 chars): ...{r['prompt'][-200:]!r}")
    print(f"GOLD: {r['gold']!r}")
    print(f"COMPLETION (first 400 chars): {r['completion'][:400]!r}")
    print()

In [ ]:
print("Eval complete. Results saved to /kaggle/working/eval_results.csv + eval_summary.json")